## Skieur Mapping Analysis: MP1 vs MP2 (SS pickles)

**Fixed pairing: `tracking_only` (MP1) -> `mapping_change` (MP2)**
Each pair is two phases of the same physical recording, linked by `paired_session` in Google Sheet.

| Session type | Condition | Role |
|-------------|-----------|------|
| `tracking_only` | Tracking only | MP1 — animal controls frequency, no playback |
| `mapping_change` | Tracking + Playback | MP2 — mapping changed, both conditions |

Google Sheet is the single source of truth. Each feature DataFrame gets `mapping_type`, `session_type`, `condition_label` columns.


In [1]:
# ============================================================
# Imports + Config
# ============================================================
import os, sys, pickle, gc, subprocess
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_rel, ttest_ind, chi2
from scipy.ndimage import gaussian_filter
from tqdm.auto import tqdm

from utils_load_data import *
from utils_trajectories import *

# ---- Paths ----
save_directory = "./"
NAS = r"\\\\129.199.81.18\\\\data5\\\\eTheremin"

# ---- Parameters ----
dt = 0.005
t_pre, t_post = 0.3, 0.3
time = np.arange(-t_pre, t_post + dt, dt)
n_pre = int(t_pre/dt - 1)

# ---- Plotting ----
mpl.rcdefaults()
plt.rcParams.update({
    'font.size': 7, 'axes.linewidth': 0.5,
    'axes.spines.top': False, 'axes.spines.right': False,
    'xtick.major.width': 0.5, 'ytick.major.width': 0.5,
    'xtick.major.size': 2, 'ytick.major.size': 2,
    'xtick.direction': 'out', 'ytick.direction': 'out',
    'pdf.fonttype': 42, 'ps.fonttype': 42,
})

# ---- Google Sheet (session type authority) ----
SHEET_URL = ("https://docs.google.com/spreadsheets/d/"
             "1sFatSTXO0j3OONKstz7YN-mM04kNMjk_r7zo951yicU/"
             "gviz/tq?tqx=out:csv&sheet=SKIEUR")

# Use curl to avoid pandas timeout on slow connections
try:
    df_sheet = pd.read_csv(SHEET_URL)
except Exception:
    import io
    result = subprocess.run(
        ["curl", "-s", "--connect-timeout", "10", SHEET_URL],
        capture_output=True, text=True)
    df_sheet = pd.read_csv(io.StringIO(result.stdout))

df_sheet = df_sheet[df_sheet["use"] == "yes"].copy()
df_sheet["session_name"] = df_sheet["session"].str.strip()

# Normalize column name (may be 'paired_session' or 'paired_sessions')
if 'paired_session' in df_sheet.columns:
    df_sheet.rename(columns={'paired_session': 'paired_sessions'}, inplace=True)

print("Session types in Google Sheet (use=yes only):")
for t in sorted(df_sheet["type"].unique()):
    names = df_sheet[df_sheet["type"] == t]["session_name"].tolist()
    print(f"  {t}: {len(names)} sessions  e.g. {names[:3]}")

print("\\nImports and config ready.")


Session types in Google Sheet (use=yes only):
  mapping_change: 13 sessions  e.g. ['SKIEUR_20260316_SESSION_02', 'SKIEUR_20260317_SESSION_03', 'SKIEUR_20260318_SESSION_01']
  mapping_change_only: 35 sessions  e.g. ['SKIEUR_20260331_SESSION_01', 'SKIEUR_20260331_SESSION_03', 'SKIEUR_20260401_SESSION_00']
  playback: 14 sessions  e.g. ['SKIEUR_20260309_SESSION_00', 'SKIEUR_20260309_SESSION_01', 'SKIEUR_20260310_SESSION_00']
  silent: 12 sessions  e.g. ['SKIEUR_20260226_SESSION_02', 'SKIEUR_20260227_SESSION_00', 'SKIEUR_20260227_SESSION_01']
  tracking_only: 14 sessions  e.g. ['SKIEUR_20260316_SESSION_01', 'SKIEUR_20260317_SESSION_00', 'SKIEUR_20260317_SESSION_02']
\nImports and config ready.


In [2]:
# ============================================================
# Helpers: load SS pickles + label from Google Sheet
# ============================================================

def load_pickled_ss(file_prefix, session_type, dt):
    data_path = os.path.join(NAS, f"{file_prefix}_{session_type}_{dt}_data_ss")
    feat_path = os.path.join(NAS, f"{file_prefix}_{session_type}_{dt}_feature_ss")
    with open(data_path, "rb") as f:
        n_data = pickle.load(f)
    with open(feat_path, "rb") as f:
        f_data = pickle.load(f)
    return n_data, f_data


def label_sessions(f_data_list, mapping_type, session_type_label):
    "Add mapping_type, session_type, condition_label to each session's features."
    for i, fd in enumerate(f_data_list):
        fd['mapping_type'] = mapping_type
        fd['session_type'] = session_type_label
        if 'Condition' in fd.columns:
            fd['condition_label'] = fd['Condition'].map(
                {0.0: 'Tracking', 1.0: 'Playback', 0: 'Tracking', 1: 'Playback'}).fillna('Unknown')
    return f_data_list


def print_classification(f_data_list, label):
    "Print mapping_type x session_type x condition_label summary."
    all_fd = pd.concat([fd[['mapping_type', 'session_type', 'condition_label']]
                         for fd in f_data_list], ignore_index=True)
    print(f"\n  {label} Classification:")
    for (mt, st, cl), count in all_fd.groupby(
        ['mapping_type', 'session_type', 'condition_label']).size().items():
        print(f"    {mt:5s} | {st:20s} | {cl:9s} : {count:>8,} timepoints")


print("Helpers ready.")


Helpers ready.


---
## Load & Merge Data

Loads both SS pickles, pairs sessions via Google Sheet `paired_session` column, and merges along the time axis.
Orphan sessions (missing partner) are dropped with a warning.


In [ ]:
# ============================================================
# Session types (fixed pairing)
# ============================================================
# tracking_only and mapping_change are strictly paired via
# the "paired_session" column in Google Sheet.
# SESSION_01 (tracking_only, TR only) + SESSION_02 (mapping_change, TR+PB)
# are two phases of the same recording — merged along the time axis.
B_MP1_TYPE = 'tracking_only'
B_MP2_TYPE = 'mapping_change'

mp1_label = 'MP1 (tracking_only)'
mp2_label = 'MP2 (mapping_change)'

for label, stype in [(mp1_label, B_MP1_TYPE), (mp2_label, B_MP2_TYPE)]:
    n = len(df_sheet[df_sheet['type'] == stype])
    print(f"  {label}: {n} sessions in Google Sheet")


In [ ]:
# ============================================================
# Load SS pickles + merge
# ============================================================

NEED_MERGE = (B_MP1_TYPE == 'tracking_only' and
              B_MP2_TYPE in ('mapping_change', 'mapping_change_only'))

print(f"Loading {mp1_label}...")
n_data_mp1_raw, f_data_mp1_raw = load_pickled_ss("SKIEUR_hs_0", B_MP1_TYPE, dt)
print(f"  Loaded {len(n_data_mp1_raw)} sessions")

print(f"Loading {mp2_label}...")
n_data_mp2_raw, f_data_mp2_raw = load_pickled_ss("SKIEUR_hs_0", B_MP2_TYPE, dt)
print(f"  Loaded {len(n_data_mp2_raw)} sessions")

if NEED_MERGE:
    # ── Pair by Google Sheet "paired_session" column ──
    # Check BOTH sides: MP1→paired_session and MP2→paired_session
    names_mp1 = df_sheet[df_sheet["type"] == B_MP1_TYPE]["session_name"].tolist()
    names_mp2 = df_sheet[df_sheet["type"] == B_MP2_TYPE]["session_name"].tolist()

    pairs = []
    seen_mp1 = set()
    # Direction 1: MP1 row has paired_session → MP2
    for _, row in df_sheet[df_sheet["type"] == B_MP1_TYPE].iterrows():
        mp1 = row["session_name"]
        mp2 = str(row.get("paired_sessions", "")).strip()
        if mp2 and mp2 != 'nan' and mp2 in names_mp2:
            pairs.append((mp1, mp2))
            seen_mp1.add(mp1)
    # Direction 2: MP2 row has paired_session → MP1 (reverse direction)
    for _, row in df_sheet[df_sheet["type"] == B_MP2_TYPE].iterrows():
        mp2 = row["session_name"]
        mp1 = str(row.get("paired_sessions", "")).strip()
        if mp1 and mp1 != 'nan' and mp1 in names_mp1 and mp1 not in seen_mp1:
            pairs.append((mp1, mp2))
            seen_mp1.add(mp1)

    # Report orphans
    seen_mp2 = set(n2 for _, n2 in pairs)
    for n in names_mp1:
        if n not in seen_mp1:
            print(f"  ORPHAN MP1: {n}")
    for n in names_mp2:
        if n not in seen_mp2:
            print(f"  ORPHAN MP2: {n}")

    print(f"\n  Pairing result: {len(pairs)} valid pairs")
    for n1, n2 in pairs[:10]:
        print(f"    {n1}  ->  {n2}")
    if len(pairs) > 10:
        print(f"    ... and {len(pairs)-10} more")

    # Filter pickle data to only sessions in valid pairs
    mp1_keep_idx = [names_mp1.index(n1) for n1, _ in pairs
                    if n1 in names_mp1 and names_mp1.index(n1) < len(n_data_mp1_raw)]
    mp2_keep_idx = [names_mp2.index(n2) for _, n2 in pairs
                    if n2 in names_mp2 and names_mp2.index(n2) < len(n_data_mp2_raw)]
    n_min = min(len(mp1_keep_idx), len(mp2_keep_idx))
    mp1_keep_idx = mp1_keep_idx[:n_min]
    mp2_keep_idx = mp2_keep_idx[:n_min]

    n_data_mp1 = [n_data_mp1_raw[i] for i in mp1_keep_idx]
    f_data_mp1 = [f_data_mp1_raw[i] for i in mp1_keep_idx]
    n_data_mp2 = [n_data_mp2_raw[i] for i in mp2_keep_idx]
    f_data_mp2 = [f_data_mp2_raw[i] for i in mp2_keep_idx]

    # Merge MP1 + MP2 along time axis (MP1 first, then MP2)
    print(f"\n  MERGING {len(n_data_mp1)} paired sessions...")
    n_data_merged, f_data_merged = [], []
    for i in range(len(n_data_mp1)):
        assert n_data_mp1[i].shape[0] == n_data_mp2[i].shape[0], \
            f"Neuron count mismatch at pair {i}"
        n_merged = np.concatenate([n_data_mp1[i], n_data_mp2[i]], axis=1)
        f_data_mp1[i]['mapping_type'] = 'MP1'
        f_data_mp2[i]['mapping_type'] = 'MP2'
        for fd, st in [(f_data_mp1[i], B_MP1_TYPE), (f_data_mp2[i], B_MP2_TYPE)]:
            fd['session_type'] = st
            if 'Condition' in fd.columns:
                fd['condition_label'] = fd['Condition'].map(
                    {0.0: 'Tracking', 1.0: 'Playback', 0: 'Tracking', 1: 'Playback'}).fillna('Unknown')
        f_data_merged.append(pd.concat([f_data_mp1[i], f_data_mp2[i]], ignore_index=True))
        n_data_merged.append(n_merged)

    n_data_all = n_data_merged
    f_data_all = f_data_merged
    MERGED = True
    print_classification(f_data_all, f"Merged ({len(pairs)} pairs)")
else:
    print("ERROR: unexpected session type combination.")


In [ ]:
# ============================================================
# Session list: Beginner/Expert split
# ============================================================
n_sess = len(n_data_all)
n_half = n_sess // 2
sessions_mp1 = df_sheet[df_sheet["type"] == B_MP1_TYPE]["session_name"].tolist()[:n_sess]
sessions_mp2 = df_sheet[df_sheet["type"] == B_MP2_TYPE]["session_name"].tolist()[:n_sess]
print(f"  Merged pairs ({n_sess} sessions):")
print(f"    Beginner ({n_half}):")
for i in range(n_half):
    print(f"      [{i}] {sessions_mp1[i]} + {sessions_mp2[i]}")
print(f"    Expert ({n_sess - n_half}):")
for i in range(n_half, n_sess):
    print(f"      [{i}] {sessions_mp1[i]} + {sessions_mp2[i]}")


---
## Build Trajectories

H1/H2 PSTH trajectories, baseline-subtracted, averaged across triggers.


In [ ]:
# ============================================================
# Build trajectories (merged data)
# ============================================================
n_data_proc = remove_average(smooth_data(n_data_all))
n_data_list, f_data_list = re_organise_data([n_data_proc], [f_data_all])
n_before = len(n_data_list)
n_data_list, f_data_list = zip(*[(n, f) for n, f in zip(n_data_list, f_data_list)
                                  if abs(n.shape[-1] - len(f)) <= 2])
n_data_list, f_data_list = list(n_data_list), list(f_data_list)
if len(n_data_list) != n_before:
    print(f"  Shape-filter dropped {n_before - len(n_data_list)} sessions")

n_half = len(n_data_list) // 2
beg_n, exp_n = list(n_data_list[:n_half]), list(n_data_list[n_half:])
beg_f, exp_f = list(f_data_list[:n_half]), list(f_data_list[n_half:])

traj_data = {}
for mt in ['MP1', 'MP2']:
    traj = {}
    for q in [(0.0, 0.5), (0.5, 1.0)]:
        track_list, pb_list = [], []
        for nd, fd in zip(beg_n + exp_n, beg_f + exp_f):
            mask = fd['mapping_type'] == mt
            if not mask.any():
                continue
            fd_sub = fd[mask].reset_index(drop=True)
            r = extract_traj_subset([nd], [fd_sub], t_pre, t_post, dt,
                                     overlap_thresh=1.0, n_pre=n_pre, full=True,
                                     trial_start=q[0], trial_end=q[1])
            track_list.extend(r[0])
            pb_list.extend(r[1])
        traj[q] = {'track': track_list, 'pb': pb_list}
    traj_data[mt] = traj
    n_tr = sum(1 for t in traj[(0.0, 0.5)]['track'] if t is not None)
    n_pb = sum(1 for t in traj[(0.0, 0.5)]['pb'] if t is not None)
    print(f"  {mt}: {n_tr} TR + {n_pb} PB traj")

traj_by_half_mp1 = traj_data['MP1']
traj_by_half_mp2 = traj_data['MP2']
print(f"\\ntraj_by_half_mp1 (MP1), traj_by_half_mp2 (MP2) ready.")
print("  NOTE: mapping_type is within-session (same neurons across MP1/MP2).")


---
## Quick Tests + LMM

Identical to `Skieur_Full_LMM.ipynb` sections 2.1–2.5.


In [ ]:
# ===============================================================
# Quick tests: Track vs PB, MP1 vs MP2
# ===============================================================
import importlib
import lmm_analysis
importlib.reload(lmm_analysis)
from lmm_analysis import build_lmm_dataframe
from scipy.stats import ttest_rel, ttest_ind

all_p_vals_anova = []
all_p_labels_anova = []

for metric in ["mean", "peak"]:
    for t_window, label in [(0.1, "0-100ms"), (0.3, "0-300ms")]:
        print(f"\n{'='*60}")
        print(f"  METRIC: {metric}  |  WINDOW: {label}")
        print("="*60)

        df1 = build_lmm_dataframe(traj_by_half_mp1, "MP1", time,
                                   auc_t_start=0.0, auc_t_end=t_window,
                                   split_half=False, metric=metric)
        df2 = build_lmm_dataframe(traj_by_half_mp2, "MP2", time,
                                   auc_t_start=0.0, auc_t_end=t_window,
                                   split_half=False, metric=metric)
        df = pd.concat([df1, df2], ignore_index=True)

        tr = df[df["condition"] == "Track"]["response"].values
        pb = df[df["condition"] == "Playback"]["response"].values
        t_tr_pb, p_tr_pb = ttest_rel(pb, tr)
        print(f"  Track vs Playback (paired): t={t_tr_pb:.3f}, p={p_tr_pb:.4e}")
        all_p_vals_anova.append(p_tr_pb)
        all_p_labels_anova.append(f"Track-vs-PB_{metric}_{label}")

        df_neuron = df.groupby(["neuron_id", "mapping"])["response"].mean().reset_index()
        mp1_vals = df_neuron[df_neuron["mapping"] == "MP1"]["response"].values
        mp2_vals = df_neuron[df_neuron["mapping"] == "MP2"]["response"].values
        t_mp, p_mp = ttest_ind(mp1_vals, mp2_vals, equal_var=False)
        print(f"  MP1 vs MP2 (Welch):        t={t_mp:.3f}, p={p_mp:.4e}")
        all_p_vals_anova.append(p_mp)
        all_p_labels_anova.append(f"MP1-vs-MP2_{metric}_{label}")

# Global FDR
from statsmodels.stats.multitest import multipletests
if all_p_vals_anova:
    reject, p_fdr, _, _ = multipletests(all_p_vals_anova, method='fdr_bh')
    print(f"\n{'='*70}")
    print("  QUICK TESTS: GLOBAL FDR (BH)")
    print("="*70)
    for label, p_raw, p_corr in zip(all_p_labels_anova, all_p_vals_anova, p_fdr):
        sig = '***' if p_corr<0.001 else ('**' if p_corr<0.01 else ('*' if p_corr<0.05 else 'n.s.'))
        print(f"  {label:<40s} raw={p_raw:.6f}  fdr={p_corr:.6f}  {sig}")


In [ ]:
# ================================================================
# Full LMM: condition * mapping * expertise * half
# ================================================================
import importlib
import lmm_analysis
importlib.reload(lmm_analysis)
from lmm_analysis import run_lmm_analysis, compare_random_effect_structures

for metric in ["mean", "peak"]:
    for t_window, label in [(0.1, "0-100ms"), (0.3, "0-300ms")]:
        print(f"\n{'='*70}")
        print(f"  METRIC: {metric}  |  WINDOW: {label}")
        print("="*70)
        result_lmm, df_lmm = run_lmm_analysis(
            traj_by_half_mp1, traj_by_half_mp2, time, save_directory,
            auc_t_start=0.0, auc_t_end=t_window, metric=metric
        )
        compare_random_effect_structures(df_lmm)


In [ ]:
# ================================================================
# Peak 0-200ms: Full LMM
# ================================================================
import importlib
import lmm_analysis
importlib.reload(lmm_analysis)
from lmm_analysis import run_lmm_analysis, compare_random_effect_structures

result_peak, df_peak = run_lmm_analysis(
    traj_by_half_mp1, traj_by_half_mp2, time, save_directory,
    auc_t_start=0.0, auc_t_end=0.2, metric="peak"
)
compare_random_effect_structures(df_peak)
